# 1. API Key 로드

In [1]:
from dotenv import load_dotenv

load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

# 1. 프롬프트 엔지니어링에서 기본적으로 생각해야할 점 

1. GPT 모델의 출력은 입력에 대한 의존도가 매우 높음
    - 잘 입력해도 원하는 결과가 나오지 않는다면
        - 입력이 모호하거나 응답에 필요한 내용이 빠졌을 수도 있음 
        - 모델의 성능으로 해결하기 힘든 요청일 수도 있음 

2. 자연어 질의를 기반으로하기 때문에 절대적으로 성능 좋은 prompt를 단정지을 수 없음 
    - 입력 문장의 문맥을 파악하고 새로운 문장을 생성할 때 마다 내부 연산이 달라질 수 있음 
    - 모델과의 대화에서 성능이 좋다는 것은 서비스를 사용하는 사용자의 만족도를 의미하는데, 이는 매우 주관적이고 판단하기 힘듦
    - 즉, 프롬프트 엔지니어링은 task에 맞는 여러번의 테스트가 필수적이며 이를 통한 반복적인 개선이 수반돼야 함

# 2. 평가 진행 방법 

1. **Human Based Evaluation** - 사람이 직접 평가하는 방식 
2. **Model Based Evaluation** - LLM이 평가하는 방식 
3. **Code Based Evaluation** - 코드와 지표로 평가하는 방식 

## 2.1 Human Based Evaluation

- 전문가 블라인드 테스트(각기 다른 LLM의 여러 답변 중에서 더 좋은 답변을 사람이 선택)
- 명확한 결과로 성능을 판단하기 쉬움
- 많은 인력에 따른 비용과 시간 요구됨
- LMSys사의 Chatbot Arena평가
    - 대표적인 HumanBased 평가 방법 중 하나로 동일한 질문에 대해 2개의 모델의 답변을 보고 승/패/무 투표 이후 모델명을 공개하는 방식
    - 사이트: https://chat.lmsys.org/

## 2.2 Model Based Evaluation 

고성능 LLM을 통해 평가하는 방법(일반적으로 GPT-4o 이상급)
- 실제로 사람이 평가하는 것과 굉장히 유사한 성능이라는 논문 결과들이 나오고 있음 
- 3가지의 평가 방식
    1. Pairwise Comparison
        - 평가를 받을 2개의 모델에 같은 질문을 하고, 고성능 모델이 2개의 답변을 받아 둘 중 어떤 답변이 더 좋은지 또는 무승부인지 출력
    2. Single Answer Grading
        - 질문과 답변이 있을 때 답변에 점수를 매기는 것 
    3. Reference-Guided Grading 
        - 예시 답변을 주고 이와 비교하여 +, -로 상대적인 점수를 매기는 것

## 2.3 Code Based Evaluation 

- 우리에게 익숙한 코드/로직을 통한 평가법
    - Accuracy, Precision, Recall, F1Score
    - ROUGE(Recall-Oriented Understudy for Gisting Evaluation): 요약 및 자연어 생성을 평가하는 지표 
    - BLUE(Bilingual Evaluation Understudy): 번역, 지연어 생성 등을 평가
- 단, HumanBased나 Model Based에 비해 실제 사용자의 만족과는 다소 거리가 있을 수 있음 


## 2.4 그 외: LLM Benchmark 플랫폼 - 인공지능 모델, 특히 LLM의 성능을 평가하기 위해 설계된 시스템 

- **MT-bench**
    - multi-turn 대화 능력을 평가하는 벤치마크(평가시스템), 사용자의 지시를 정확하게 따르고 여러 차례의 이어지는 대화에서 일관된 응답을 제공하는 모델의 능력을 테스트
    - 먼저 58명의 전문가가 모델의 응답을 평가하고 LLM을 심판으로 사용하여 사람의 평가와 일치하는지 검증하는 방식
    - multi-turn 대화: 한 번의 응답이 아니라 이어지는 여러 질문들에 얼마나 잘 응답하는지를 평가하기 위해 multi-turn 대화방식으로 평가.\
    MT-bench는 8개의 카테고리 (작문, 역할놀이 추출, 추론, 수학, 코딩, STEM(과학, 기술, 공학, 수학), 인문 및 사회과학)의 80개의 고품질 질문으로 구성되어 있고 각 질문은 여러 차례의 응답을 요구하여 모델의 대화 지속능력을 평가

# 3. 모델 성능 평가 

- 위의 평가 방법 중 ModelBased 방식 중 1번(Pairwise Comparison)에 해당하는 방식으로 모델 성능평가 진행 
- 관련 논문의 평가용 프롬프트 사용
- gpt-3.5와 gpt-4o의 응답을 비교하여 gpt-5-nano에 더 나은 응답이 뭔지 질의하는 방법으로 진행 

## 3.1 다른 버전의 GPT모델 성능 비교

In [8]:
# gpt-3.5 응답 생성
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

model_3 = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)  # 모델 성능평가시에는 temperature=0!!!!

question = "하늘은 왜 파란색인가요?"
response_3 = model_3.invoke(question).content
print(response_3)

하늘이 파란색인 이유는 태양빛이 대기 중의 공기 분자들과 상호 작용하여 발생하는 현상 때문입니다. 태양빛은 다양한 색상의 빛으로 구성되어 있지만, 대기 중의 공기 분자들은 파란색 빛을 더 많이 흡수하고 다른 색상의 빛을 덜 흡수합니다. 이러한 현상으로 인해 하늘은 파란색으로 보이게 됩니다.


In [9]:
# gpt-4o 응답 생성

model_4 = ChatOpenAI(model="gpt-4o", temperature=0)  # 모델 성능평가시에는 temperature=0!!!!

question = "하늘은 왜 파란색인가요?"
response_4 = model_4.invoke(question).content
print(response_4)

하늘이 파란색으로 보이는 이유는 주로 레일리 산란(Rayleigh scattering) 때문입니다. 태양빛은 여러 가지 색의 빛으로 구성되어 있으며, 이 빛은 서로 다른 파장을 가지고 있습니다. 대기 중의 분자와 입자들은 이 빛을 산란시키는데, 파장이 짧은 파란색 빛이 다른 색의 빛보다 더 많이 산란됩니다.

태양빛이 대기를 통과할 때, 파란색 빛은 모든 방향으로 산란되어 하늘 전체에 퍼지게 됩니다. 그래서 우리가 하늘을 볼 때, 여러 방향에서 산란된 파란색 빛이 눈에 들어와 하늘이 파랗게 보이는 것입니다. 반면, 파장이 긴 빨간색 빛은 산란이 덜 되어 주로 직진합니다. 이러한 이유로 해가 지거나 뜰 때, 태양빛이 더 두꺼운 대기층을 통과하게 되면 파란색 빛은 더 많이 산란되고, 빨간색과 주황색 빛이 더 많이 남아 하늘이 붉게 보이게 됩니다.


In [10]:
# 평가용 프롬프트 

prompt = f"""\
[System]
Please act as an impartial judge and evaluate the quality of the responses provided by two
AI assistants to the user question displayed below. You should choose the assistant that
follows the user's instructions and answers the user's question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.

[User Question]
{question}

[The Start of Assistant A's Answer]
{response_3}
[The End of Assistant A's Answer]

[The Start of Assistant B's Answer]
{response_4}
[The End of Assistant B's Answer]
"""

print(prompt)

[System]
Please act as an impartial judge and evaluate the quality of the responses provided by two
AI assistants to the user question displayed below. You should choose the assistant that
follows the user's instructions and answers the user's question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.

[User Question]
하늘은 왜 파란색인가요?

[The Start of Assist

In [11]:
# 평가용 프롬프트를 모델에 전달하여 두 응답 비교 및 평가

evaluator_model = ChatOpenAI(model="gpt-5-nano", temperature=0)
evaluation_res = evaluator_model.invoke(prompt).content
print(evaluation_res)

비교 요약:
- Assistant A: 하늘이 파란색인 이유를 대기 분자들이 파란빛을 흡수한다는 잘못된 주장으로 설명합니다. 실제로는 파란 빛이 더 많이 산란되어 눈에 들어오기 때문에 하늘이 파랗습니다. 이 부분이 과학적 정확성에서 벗어나 있습니다. 나머지 설명은 다소 일반적이고 구체성이 부족합니다.
- Assistant B: 레일리 산란(Rayleigh scattering) 원리를 명확히 제시하고, 파장이 짧은 파란 빛이 더 많이 산란된다는 점을 구체적으로 설명합니다. 또한 해가 질 때의 현상까지 연결해 상황별 예시를 제공합니다. 과학적으로 정확하고 깊이가 있습니다.

따라서 B가 더 정확하고 도움이 된다.

[[B]]


## 3.2 다른 회사의 모델 성능 비교

OpenAI의 GPT와 Goole의 Gemini의 성능을 비교해보자

In [1]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

### 3.2.1 두 개의 모델로부터 응답 생성

In [2]:
# 투자 포트폴리오 다각도 분석 함수 정의 
def portfolio_strategy_analysis(model, portfolio):

    # 보수적 투자 전략 프롬프트
    prompt_conservative = ChatPromptTemplate([
        ("system", "당신은 보수적 투자 전략 전문가입니다.\n포트폴리오 안정성 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
        ("user", "{input_data}")
    ])
    # 공격적 투자 전략 프롬프트
    prompt_aggressive = ChatPromptTemplate([
        ("system", "당신은 공격적 투자 전략 전문가입니다.\n포트폴리오 수익률 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
        ("user", "{input_data}")
    ])

    # 균형적 투자 전략 프롬프트
    prompt_balanced = ChatPromptTemplate([
        ("system", "당신은 균형적 투자 전략 전문가입니다.\n포트폴리오 리스크-수익 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
        ("user", "{input_data}")
    ])

    # 3개의 chain 생성
    chain_conservative = prompt_conservative | model | StrOutputParser()
    chain_aggressive = prompt_aggressive | model | StrOutputParser()
    chain_balanced = prompt_balanced | model | StrOutputParser()

    # 병렬 실행
    runnable_parallel = RunnableParallel({
        "보수형 전략": chain_conservative,
        "공격형 전략": chain_aggressive,
        "균형 전략": chain_balanced
    })

    result_portfolio = runnable_parallel.invoke(portfolio)

    return result_portfolio


In [3]:
portfolio = "국내주식 70%, 해외주식 20%, 현금 10%"

In [4]:
# gpt 응답 생성
model_gpt = ChatOpenAI(model="gpt-5-nano")
gpt_response = portfolio_strategy_analysis(model_gpt, portfolio)

# gemini 응답 생성
model_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
gemini_response = portfolio_strategy_analysis(model_gemini, portfolio)

### 3.2.2 응답 비교 및 성능 평가 

In [6]:
# 평가용 프롬프트 

prompt = f"""\
[System]
Please act as an impartial judge and evaluate the quality of the responses provided by two
AI assistants to the user question displayed below. You should choose the assistant that
follows the user's instructions and answers the user's question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.

[User Question]
{portfolio}

[The Start of Assistant A's Answer]
{gpt_response}
[The End of Assistant A's Answer]

[The Start of Assistant B's Answer]
{gemini_response}
[The End of Assistant B's Answer]
"""

print(prompt)

[System]
Please act as an impartial judge and evaluate the quality of the responses provided by two
AI assistants to the user question displayed below. You should choose the assistant that
follows the user's instructions and answers the user's question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.

[User Question]
국내주식 70%, 해외주식 20%, 현금 10%

[The St

In [7]:
# 평가용 프롬프트를 모델에 전달하여 두 응답 비교 및 평가

eval_model = ChatOpenAI(model="gpt-5.2", temperature=0)
eval_res = eval_model.invoke(prompt).content
print(eval_res)

두 답변 모두 사용자가 제시한 자산배분(국내주식 70%, 해외주식 20%, 현금 10%)에 대해 ‘보수/균형/공격’ 관점의 코멘트를 제공하려는 형태입니다. 그러나 **Assistant A**는 각 전략별로 **구체적인 재조정 비율(예시 수치)**을 제시하고, 국내 편중·환율리스크·현금의 인플레이션 리스크 등 **핵심 리스크 요인**을 짚은 뒤 **리밸런싱 주기/규칙, ETF 활용 등 실행 팁**까지 제공합니다. 사용자가 추가 정보를 주지 않은 상황에서도 의사결정에 도움이 되는 실질적 제안이 포함되어 있습니다.

반면 **Assistant B**는 전반적으로 “해외 비중을 늘리고 현금을 조정하라”는 **일반론을 반복**하며, 각 전략 간 차별성이 약하고 **구체적인 배분안이나 실행 방법**이 없어 사용자의 질문(자산배분에 대한 평가/조정 제안)에 비해 실용성이 떨어집니다. 또한 보수형에서조차 해외 비중 확대를 강조하는 등 메시지가 다소 모호합니다.

따라서 사용자의 의도(포트폴리오 평가 및 조정 방향 제시)에 더 직접적으로 부합하고 실행 가능한 정보를 제공한 **Assistant A**가 더 우수합니다.

[[A]]


# 4. 장단점 비교

1. Human Based Evaluation
    - 통제된 환경을 가정했을 때 사람이 직접 평가하는 방법이라 안정적이고 신뢰가 높음 
    - 다만 평가하는 인원이 불특정 다수일 경우 약간의 노이즈가 발생할 수 있음 
    - 전문 분야의 경우 해당 전문가가 아닌 일반인이 평가할 경우 정확도 및 속도가 낮아질 수 있음

2. Model Based Evaluation
    - 사람의 평가와 어느정도 유사한 수준의 평가를 내릴 수 있음
    - 평가를 위해 API 호출 횟수 및 토큰의 수가 늘어나는데 이는 평가 데이터가 굉장히 많다면 수백만원 이상은 금방 넘어갈 수 있어서 비용에 대한 부분을 생각해야 함

3. Code Based Evaluation 
    - 위 방법들에 대해서 비용이 훨씬 적지만 task에 따라서 활용할 수 있는 범위가 제한적
    - 사람이 만족할만한 답변을 선택하는데 있어서 신뢰도가 상대적으로 떨어지는 편 

# 5. 결론
- 각 task에 적합한 전문 인력들이 평가하는 방법이 가장 좋음
- 그러나 현실적, 효율성 문제로 모델이 평가하는 방법도 충분히 좋음
- 정량적 평가와 정성적 평가를 모두 진행하는 게 가장 이상적인 케이스
- 실 서비스로 봤을 때 언어 모델의 최종 평가 지표는 결국 **사용자의 만족**이 가장 중요함